In [9]:
import pandas as pd
import numpy as np
import os
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, r2_score

# Load data
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')

regions = ['Rakhiyal', 'Bopal', 'Ambawadi', 'Chandkheda', 'Vastral']

# Filter train and test data (hourly)
train_df = df[(df['datetime'].dt.year >= 2019) & (df['datetime'].dt.year <= 2023)].copy()
test_df = df[df['datetime'].dt.year == 2024].copy()

# Prepare output folder for 2025 predictions
output_folder_2025 = 'predictions_2025'
os.makedirs(output_folder_2025, exist_ok=True)

current_dir = os.getcwd()
metrics_list = []

for region in regions:
    print(f"\nProcessing region: {region}")

    # Prepare hourly training and test series with interpolation
    train_series = train_df.set_index('datetime')[region].interpolate(method='time')
    test_series = test_df.set_index('datetime')[region].interpolate(method='time')

    # Fit Holt-Winters on hourly data with daily seasonality (24 hours)
    hw_model = ExponentialSmoothing(
        train_series,
        trend='add',
        seasonal='add',
        seasonal_periods=24,
        initialization_method='estimated'
    ).fit(optimized=True)

    # Forecast for test period (2024)
    hw_preds = hw_model.forecast(len(test_series))

    # Calculate error metrics
    mse = mean_squared_error(test_series, hw_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(test_series, hw_preds)

    print(f"  2024 MSE: {mse:.4f}")
    print(f"  2024 RMSE: {rmse:.4f}")
    print(f"  2024 R2 Score: {r2:.4f}")

    # Prepare 2024 prediction DataFrame with hourly granularity
    pred_2024_df = pd.DataFrame({
        'date': test_series.index.date,
        'hour': test_series.index.hour,
        'predicted_temperature': hw_preds.values,
        'actual_temperature': test_series.values
    })

    filename_2024 = f"holtwinters_hourly_{region.lower()}_2024.csv"
    pred_2024_df.to_csv(os.path.join(current_dir, filename_2024), index=False)
    print(f"Saved 2024 Holt-Winters hourly predictions for {region} as {filename_2024}")

    # Forecast 2025 hourly
    date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')
    forecast_2025 = hw_model.forecast(len(date_range_2025))

    pred_2025_df = pd.DataFrame({
        'date': date_range_2025.date,
        'hour': date_range_2025.hour,
        'predicted_temperature': forecast_2025.values
    })

    filename_2025 = f"holtwinters_hourly_{region.lower()}_2025.csv"
    pred_2025_df.to_csv(os.path.join(output_folder_2025, filename_2025), index=False)
    print(f"Saved 2025 Holt-Winters hourly predictions for {region} as {os.path.join(output_folder_2025, filename_2025)}")

    # Store metrics
    metrics_list.append({
        'region': region,
        'mse_2024': mse,
        'rmse_2024': rmse,
        'r2_2024': r2
    })

# Save metrics CSV
metrics_df = pd.DataFrame(metrics_list)
metrics_filename = 'holtwinters_hourly_model_metrics_2024.csv'
metrics_df.to_csv(os.path.join(current_dir, metrics_filename), index=False)
print(f"\nSaved Holt-Winters hourly error metrics for all regions as {metrics_filename}")



Processing region: Rakhiyal
  2024 MSE: 174.1684
  2024 RMSE: 13.1973
  2024 R2 Score: -3.9718
Saved 2024 Holt-Winters hourly predictions for Rakhiyal as holtwinters_hourly_rakhiyal_2024.csv
Saved 2025 Holt-Winters hourly predictions for Rakhiyal as predictions_2025\holtwinters_hourly_rakhiyal_2025.csv

Processing region: Bopal
  2024 MSE: 415.7954
  2024 RMSE: 20.3911
  2024 R2 Score: -10.5763
Saved 2024 Holt-Winters hourly predictions for Bopal as holtwinters_hourly_bopal_2024.csv
Saved 2025 Holt-Winters hourly predictions for Bopal as predictions_2025\holtwinters_hourly_bopal_2025.csv

Processing region: Ambawadi
  2024 MSE: 415.7954
  2024 RMSE: 20.3911
  2024 R2 Score: -10.5763
Saved 2024 Holt-Winters hourly predictions for Ambawadi as holtwinters_hourly_ambawadi_2024.csv
Saved 2025 Holt-Winters hourly predictions for Ambawadi as predictions_2025\holtwinters_hourly_ambawadi_2025.csv

Processing region: Chandkheda
  2024 MSE: 261.8648
  2024 RMSE: 16.1822
  2024 R2 Score: -6.2310
